In [ ]:
import numpy as np
import pandas as pd
import librosa
import os
import torch
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import Wav2Vec2Processor, Wav2Vec2Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, GRU, Dense, Dropout, Flatten, MaxPooling1D, BatchNormalization
from tensorflow.keras.activations import swish
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load Wav2Vec2.0 model
processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")
wav2vec_model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.84k [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/configuration_utils.py:311: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

In [ ]:
# Set directories
main_dir = "/content/drive/MyDrive/Colab Notebooks/5T/data"
labels = os.listdir(main_dir)
x, y = [], []

In [ ]:
%%time
for label in labels:
    sub_dir = os.path.join(main_dir, label)
    files = os.listdir(sub_dir)
    for file in files:
        path = os.path.join(sub_dir, file)
        audio, sr = librosa.load(path, sr=16000)

        # Ensure minimum length for FFT
        min_length = 2048
        if len(audio) < min_length:
            audio = np.pad(audio, (0, min_length - len(audio)), mode='constant')

        # Dynamic FFT size
        n_fft = min(2048, len(audio) // 2)
        hop_length = n_fft // 4

        # Extract MFCCs
        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=80, n_fft=n_fft, hop_length=hop_length)
        delta_mfcc = librosa.feature.delta(mfccs)
        delta2_mfcc = librosa.feature.delta(mfccs, order=2)

        features = np.vstack([mfccs, delta_mfcc, delta2_mfcc])
        mfccs_scaled = np.mean(features.T, axis=0)

        x.append(mfccs_scaled)
        y.append(label)

CPU times: user 12min 36s, sys: 29.3 s, total: 13min 6s
Wall time: 22min 14s


In [ ]:
%%time
# Encode Labels
le = LabelEncoder().fit(y)
y = le.transform(y)
np.save("AQ.npy", le.classes_)

CPU times: user 15.5 ms, sys: 976 µs, total: 16.5 ms
Wall time: 19 ms


In [ ]:
# Split Data
x_train, x_test, y_train, y_test = train_test_split(np.array(x), np.array(y), test_size=0.2, random_state=42)
x_train = np.expand_dims(x_train, axis=2)
x_test = np.expand_dims(x_test, axis=2)

In [ ]:
from tensorflow.keras.layers import GaussianNoise

# Apply Gaussian Noise to the input to improve generalization
model = Sequential()

# Convolutional Layers
model.add(Conv1D(128, 5, padding='same', input_shape=(x_train.shape[1], 1), kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(BatchNormalization())
model.add(tf.keras.layers.LeakyReLU(alpha=0.01))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.3))  # Increased dropout

model.add(Conv1D(256, 5, padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(BatchNormalization())
model.add(tf.keras.layers.LeakyReLU(alpha=0.01))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.4))  # Increased dropout

model.add(Conv1D(384, 3, padding='same', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))  # Reduced filters slightly
model.add(BatchNormalization())
model.add(tf.keras.layers.LeakyReLU(alpha=0.01))
model.add(MaxPooling1D(pool_size=2))
model.add(Dropout(0.5))  # Increased dropout

# GRU Layers
model.add(GRU(192, activation='tanh', return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.0005)))  # Reduced units slightly
model.add(GRU(128, activation='tanh', return_sequences=True, kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(GRU(96, activation='tanh', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))
model.add(Dropout(0.5))  # Increased dropout

# Fully Connected Layers
model.add(Dense(512, activation='swish', kernel_regularizer=tf.keras.regularizers.l2(0.0005)))  # L2 Regularization added
model.add(Dropout(0.6))
model.add(Dense(len(labels), activation='softmax'))


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.11/dist-packages/keras/src/layers/activations/leaky_relu.py:41: UserWarning: Argument `alpha` is deprecated. Use `negative_slope` instead.
  warnings.warn(


In [ ]:
# Optimizer with Cosine Decay
cosine_decay = tf.keras.optimizers.schedules.CosineDecay(initial_learning_rate=0.0005, decay_steps=10000, alpha=0.0001)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)

model.compile(loss='sparse_categorical_crossentropy', optimizer=optimizer, metrics=['accuracy'])

In [ ]:
# Learning Rate Scheduler
lr_scheduler = ReduceLROnPlateau(monitor='val_accuracy', patience=4, factor=0.5, min_lr=1e-5)

In [ ]:
# Train Model with Larger Batch Size
history = model.fit(x_train, y_train, batch_size=64, epochs=100, validation_data=(x_test, y_test), callbacks=[lr_scheduler])

Epoch 1/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 16s 28ms/step - accuracy: 0.1324 - loss: 3.9756 - val_accuracy: 0.1840 - val_loss: 3.4080 - learning_rate: 5.0000e-04
Epoch 2/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 6s 28ms/step - accuracy: 0.1525 - loss: 3.4148 - val_accuracy: 0.1840 - val_loss: 3.3205 - learning_rate: 5.0000e-04
Epoch 3/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 27ms/step - accuracy: 0.1687 - loss: 3.3333 - val_accuracy: 0.1840 - val_loss: 3.2960 - learning_rate: 5.0000e-04
Epoch 4/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 10s 24ms/step - accuracy: 0.1701 - loss: 3.2852 - val_accuracy: 0.1840 - val_loss: 3.2705 - learning_rate: 5.0000e-04
Epoch 5/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 5s 26ms/step - accuracy: 0.1691 - loss: 3.2725 - val_accuracy: 0.1840 - val_loss: 3.2592 - learning_rate: 5.0000e-04
Epoch 6/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - accuracy: 0.1708 - loss: 3.2412 - val_accuracy: 0.1840 - val_loss: 3.2423 - learning_rate: 2.5000e-04
Epoch 7/100
192/192 ━━━━━━━━━━━━━━━━━━━━ 6s 26ms/

# FINE TUNE

In [ ]:
# --- Step 2: Fine-Tuning ---

# Unfreeze all layers (you can customize to unfreeze specific layers too)
for layer in model.layers:
    layer.trainable = True

# Recompile with a much smaller learning rate
fine_tune_lr = 1e-5  # Fine-tuning learning rate
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(learning_rate=fine_tune_lr),
    metrics=['accuracy']
)

# Fine-tune for a few epochs
fine_tune_epochs = 20  # You can adjust
history_fine = model.fit(
    x_train, y_train,
    batch_size=32,  # Often smaller batch size for fine-tuning
    epochs=fine_tune_epochs,
    validation_data=(x_test, y_test),
    callbacks=[lr_scheduler]
)

Epoch 1/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 13s 19ms/step - accuracy: 0.9206 - loss: 0.4652 - val_accuracy: 0.9166 - val_loss: 0.4933 - learning_rate: 1.0000e-05
Epoch 2/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 9s 18ms/step - accuracy: 0.9139 - loss: 0.4719 - val_accuracy: 0.9160 - val_loss: 0.4965 - learning_rate: 1.0000e-05
Epoch 3/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step - accuracy: 0.9207 - loss: 0.4556 - val_accuracy: 0.9179 - val_loss: 0.4929 - learning_rate: 1.0000e-05
Epoch 4/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.9231 - loss: 0.4519 - val_accuracy: 0.9173 - val_loss: 0.4952 - learning_rate: 1.0000e-05
Epoch 5/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - accuracy: 0.9137 - loss: 0.4744 - val_accuracy: 0.9186 - val_loss: 0.4882 - learning_rate: 1.0000e-05
Epoch 6/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 10s 17ms/step - accuracy: 0.9223 - loss: 0.4534 - val_accuracy: 0.9163 - val_loss: 0.4932 - learning_rate: 1.0000e-05
Epoch 7/20
384/384 ━━━━━━━━━━━━━━━━━━━━ 11s 19ms/step 

In [ ]:
# --- Step 3: Save Fine-tuned Model ---

model.save("AQFT.keras")

# --- Step 4: Evaluate Fine-tuned Model ---

test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Fine-tuned Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Fine-tuned Test Loss: {test_loss:.4f}")

96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9182 - loss: 0.4947
Fine-tuned Test Accuracy: 92.05%
Fine-tuned Test Loss: 0.4816


In [ ]:
model.evaluate(x_test, y_test)

96/96 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.9182 - loss: 0.4947


[0.48161226511001587, 0.9205211997032166]

In [ ]:
# Training accuracy and loss over epochs
acc = history.history['accuracy'] # Changed detector to history
val_acc = history.history['val_accuracy'] # Changed detector to history
loss = history.history['loss'] # Changed detector to history
val_loss = history.history['val_loss'] # Changed detector to history

# Print final accuracy and loss values
print(f"Final Training Accuracy: {acc[-1] * 100:.2f}%")
print(f"Final Validation Accuracy: {val_acc[-1] * 100:.2f}%")
print(f"Final Training Loss: {loss[-1]:.4f}")
print(f"Final Validation Loss: {val_loss[-1]:.4f}")

Final Training Accuracy: 92.24%
Final Validation Accuracy: 91.53%
Final Training Loss: 0.4566
Final Validation Loss: 0.4924
